# EEG · Causal Random Forest — Multi-Trial Training & Evaluation

Trains on **multiple trials** and evaluates with **leave-one-trial-out (LOTO) CV**.

## Why trial-grouped CV matters

Consecutive windows within a trial share ≈99% of their samples — a random
train/test split leaks adjacent windows into both folds, inflating accuracy.
Grouping by trial forces the model to generalise *across* recording sessions,
which is what matters in live use.

```
Trials 0–7  ───►  Leave-One-Trial-Out CV (Optuna inner + LOTO outer)
Trials 8–9  ───►  Held-out test set  (never seen during training or tuning)
```

| Step | Trials used | Purpose |
|---|---|---|
| Optuna inner CV | 0–7, GroupKFold(8) | Hyperparameter search |
| LOTO outer CV | 0–7, LeaveOneGroupOut | Unbiased AUC / threshold |
| Calibration | full 0–7 | Isotonic probability calibration |
| **Test** | **8, 9** | **Final held-out evaluation** |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, logging, os

import numpy as np
import optuna
import pandas as pd
from mne.filter import filter_data
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DARK  = "#0a0e17"; CARD = "#111827"; EDGE = "#1f2937"
TEAL  = "#00e5cc"; CORAL = "#ff4f5e"; GOLD = "#ffc947"
LIME  = "#a8ff3e"; WHITE = "#f0f4ff"; GRAY = "#6b7280"

print("All imports OK.")

All imports OK.


## 1 · Configuration

In [2]:
FS      = 250          # sampling frequency (Hz)
SUBJECT = 9
TACHE   = "spt"
DATA_PATH = f"../data/subject{SUBJECT}/"

ALL_TRIALS   = list(range(10))   # trials 0-9
TRAIN_TRIALS = list(range(8))    # trials 0-7  → LOTO CV
TEST_TRIALS  = [8, 9]            # truly held out

N_TRIALS_OPTUNA = 30
APPLY_CLEANING  = False

print(f"Train trials : {TRAIN_TRIALS}")
print(f"Test  trials : {TEST_TRIALS}")

Train trials : [0, 1, 2, 3, 4, 5, 6, 7]
Test  trials : [8, 9]


## 2 · Load & Preprocess All Trials

Band-pass filter each trial independently (1–60 Hz), then concatenate into
one array with a `trial_id` group vector for grouped CV.

In [3]:
def load_trial(subject, tache, trial, data_path, eeg_cols=None):
    df = pd.read_csv(
        f"{data_path}/subject_{subject}_tache_{tache}_trial_{trial}.csv"
    )
    cols = [c for c in df.columns if c not in ("button", "timestamp")]
    if eeg_cols is not None:
        assert cols == eeg_cols, f"Trial {trial} channel mismatch"
    df.iloc[:, 1:-1] = filter_data(df.iloc[:, 1:-1].values.T, FS, 1, 60).T
    return df, cols


def build_windows(X: np.ndarray, y: np.ndarray, groups: np.ndarray,
                  T: int, per_window_norm: bool = True):
    """
    Causal sliding windows of length T.

    Parameters
    ----------
    X      : (N, C) raw EEG
    y      : (N,)   labels
    groups : (N,)   trial_id per sample (used to keep windows within a trial)
    T      : look-back in samples
    per_window_norm : z-score each window per channel independently

    Returns
    -------
    X_win  : (M, C*T)
    y_win  : (M,)
    g_win  : (M,)   trial group of each window (= group of sample i+T)
    """
    N, C = X.shape
    wins, labels, grps = [], [], []
    for i in range(T, N):
        if groups[i] != groups[i - T]:   # window spans two trials — skip
            continue
        w = X[i - T : i].copy()          # (T, C)
        if per_window_norm:
            mu = w.mean(axis=0, keepdims=True)
            sd = w.std(axis=0, keepdims=True) + 1e-8
            w  = (w - mu) / sd
        wins.append(w.flatten())
        labels.append(y[i])
        grps.append(groups[i])
    return (
        np.array(wins,   dtype=np.float32),
        np.array(labels, dtype=np.int64),
        np.array(grps,   dtype=np.int64),
    )


def feature_names(eeg_cols, T):
    return [f"{ch}_lag{t}" for ch in eeg_cols for t in range(T)]

print("Helper functions defined.")

Helper functions defined.


In [4]:
print("Loading train trials …")
EEG_COLS = None
X_parts, y_parts, g_parts = [], [], []

for t in TRAIN_TRIALS:
    df, cols = load_trial(SUBJECT, TACHE, t, DATA_PATH, eeg_cols=EEG_COLS)
    if EEG_COLS is None:
        EEG_COLS = cols
    X_parts.append(df[EEG_COLS].values.astype(np.float32))
    y_parts.append(df["button"].values.astype(int))
    g_parts.append(np.full(len(df), t, dtype=np.int64))
    bal = df["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df):>6,} samples  label={bal}")

X_train_raw = np.concatenate(X_parts, axis=0)
y_train_raw = np.concatenate(y_parts, axis=0)
g_train_raw = np.concatenate(g_parts, axis=0)

N_CHANNELS = len(EEG_COLS)
print(f"\nCombined train : {X_train_raw.shape}")
print(f"Class balance  : {dict(zip(*np.unique(y_train_raw, return_counts=True)))}")  

Loading train trials …
Setting up band-pass filter from 1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 60.00 Hz
- Upper transition bandwidth: 15.00 Hz (-6 dB cutoff frequency: 67.50 Hz)
- Filter length: 825 samples (3.300 s)

  trial 0: 20,338 samples  label={0: 18712, 1: 1626}
Setting up band-pass filter from 1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 60.00 Hz
- 

In [5]:
print("Loading test trials …")
X_test_parts, y_test_parts, g_test_parts = [], [], []

for t in TEST_TRIALS:
    df, _ = load_trial(SUBJECT, TACHE, t, DATA_PATH, eeg_cols=EEG_COLS)
    X_test_parts.append(df[EEG_COLS].values.astype(np.float32))
    y_test_parts.append(df["button"].values.astype(int))
    g_test_parts.append(np.full(len(df), t, dtype=np.int64))
    bal = df["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df):>6,} samples  label={bal}")

X_test_raw = np.concatenate(X_test_parts, axis=0)
y_test_raw = np.concatenate(y_test_parts, axis=0)
g_test_raw = np.concatenate(g_test_parts, axis=0)
print(f"\nCombined test  : {X_test_raw.shape}")

Loading test trials …
Setting up band-pass filter from 1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 60.00 Hz
- Upper transition bandwidth: 15.00 Hz (-6 dB cutoff frequency: 67.50 Hz)
- Filter length: 825 samples (3.300 s)

  trial 8: 15,000 samples  label={0: 12666, 1: 2334}
Setting up band-pass filter from 1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 60.00 Hz
- U

## 3 · Optuna Hyperparameter Search — GroupKFold by Trial

Inner CV uses `GroupKFold(n_splits=len(TRAIN_TRIALS))` so each fold holds out
one complete trial.  This prevents within-trial leakage during tuning.

| Hyperparameter | Range |
|---|---|
| **T** | 4 – 128 samples |
| `n_estimators` | 50 – 500 (log) |
| `max_depth` | None, 5, 10, 20, 30 |
| `min_samples_split` | 2 – 20 |
| `min_samples_leaf` | 1 – 10 |
| `max_features` | sqrt, log2, 0.1, 0.3 |
| `class_weight` | balanced, None |

In [ ]:
N_INNER_SPLITS = len(TRAIN_TRIALS)   # one fold per train trial
cv_inner = GroupKFold(n_splits=N_INNER_SPLITS)


def objective(trial):
    T = trial.suggest_int("T", 4, 128, log=True)

    n_estimators      = trial.suggest_int("n_estimators", 50, 500, log=True)
    max_depth         = trial.suggest_categorical("max_depth", [None, 5, 10, 20, 30])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.1, 0.3])
    class_weight      = trial.suggest_categorical("class_weight", ["balanced", None])

    X_win, y_win, g_win = build_windows(X_train_raw, y_train_raw, g_train_raw, T)

    clf = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        max_features=max_features, class_weight=class_weight,
        n_jobs=-1, random_state=42,
    )

    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(
            cv_inner.split(X_win, y_win, groups=g_win)):
        clf.fit(X_win[tr_idx], y_win[tr_idx])
        probs = clf.predict_proba(X_win[va_idx])[:, 1]
        auc   = roc_auc_score(y_win[va_idx], probs)
        fold_aucs.append(auc)
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best AUC   : {study.best_value:.4f}")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k:22s}: {v}")

  0%|                                                                                                                                                            | 0/30 [00:00<?, ?it/s]

## 4 · Extract Best Config & Build Full Train Windows

In [ ]:
p = study.best_params
T                 = p["T"]
n_estimators      = p["n_estimators"]
max_depth         = p["max_depth"]
min_samples_split = p["min_samples_split"]
min_samples_leaf  = p["min_samples_leaf"]
max_features      = p["max_features"]
class_weight      = p["class_weight"]

X_win, y_win, g_win = build_windows(X_train_raw, y_train_raw, g_train_raw, T)
feat_cols = feature_names(EEG_COLS, T)

print(f"Best T       : {T} samples  ({T/FS*1000:.0f} ms look-back)")
print(f"Dataset      : {X_win.shape}  →  {y_win.shape}")
print(f"Trial groups : {np.unique(g_win)}")
print(f"n_estimators : {n_estimators}")
print(f"max_depth    : {max_depth}")
print(f"min_samples_split / leaf : {min_samples_split} / {min_samples_leaf}")
print(f"max_features : {max_features}")
print(f"class_weight : {class_weight}")

## 5 · Leave-One-Trial-Out (LOTO) Outer CV

Each fold holds out **one complete trial** as validation.  
This gives an unbiased estimate of generalisation to unseen trials  
and is used to find the optimal decision threshold (Youden's J).

In [ ]:
logo = LeaveOneGroupOut()

best_clf = RandomForestClassifier(
    n_estimators=n_estimators, max_depth=max_depth,
    min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
    max_features=max_features, class_weight=class_weight,
    n_jobs=-1, random_state=42,
)

loto_aucs  = []
all_probs  = np.zeros(len(y_win), dtype=np.float32)
all_preds  = np.full(len(y_win), -1, dtype=int)

for held_trial, (tr_idx, va_idx) in zip(
        np.unique(g_win),
        logo.split(X_win, y_win, groups=g_win)):
    best_clf.fit(X_win[tr_idx], y_win[tr_idx])
    probs = best_clf.predict_proba(X_win[va_idx])[:, 1]
    auc   = roc_auc_score(y_win[va_idx], probs)
    all_probs[va_idx] = probs
    all_preds[va_idx] = (probs >= 0.5).astype(int)
    loto_aucs.append(auc)
    print(f"  held-out trial {held_trial:2d}  AUC={auc:.4f}  "
          f"n_val={len(va_idx):,}")

mask = all_preds >= 0
print(f"\nLOTO mean AUC : {np.mean(loto_aucs):.4f} ± {np.std(loto_aucs):.4f}")
print(f"Accuracy @ 0.5: {(all_preds[mask] == y_win[mask]).mean():.4f}")
print()
print(classification_report(y_win[mask], all_preds[mask], target_names=["Closed", "Open"]))

## 6 · Optimal Threshold (Youden's J) + Probability Calibration

In [ ]:
# ── Optimal threshold from LOTO holdout probabilities ─────────────────────
fpr_cv, tpr_cv, thresholds_cv = roc_curve(y_win[mask], all_probs[mask])
best_thresh = float(thresholds_cv[np.argmax(tpr_cv - fpr_cv)])
preds_opt   = (all_probs[mask] >= best_thresh).astype(int)
acc_opt     = (preds_opt == y_win[mask]).mean()

print(f"Optimal threshold (Youden's J) : {best_thresh:.3f}")
print(f"Accuracy @ optimal threshold   : {acc_opt:.4f}")
print()
print(classification_report(y_win[mask], preds_opt, target_names=["Closed", "Open"]))

# ── Refit on all train data → feature importance ───────────────────────────
print("Refitting on all train data …")
best_clf.fit(X_win, y_win)
importances = best_clf.feature_importances_

# ── Probability calibration ────────────────────────────────────────────────
print("Calibrating (isotonic, LOTO) …")
calibrated_clf = CalibratedClassifierCV(
    RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        max_features=max_features, class_weight=class_weight,
        n_jobs=-1, random_state=42,
    ),
    cv=LeaveOneGroupOut(),
    method="isotonic",
)
calibrated_clf.fit(X_win, y_win, groups=g_win)
print("Done.")

## 7 · Held-Out Test Evaluation (Trials 8 & 9)

These trials were **never seen** during hyperparameter search, CV, threshold
tuning, or calibration.  Results here reflect true out-of-distribution performance.

In [ ]:
X_win_test, y_win_test, g_win_test = build_windows(
    X_test_raw, y_test_raw, g_test_raw, T
)

probs_test = calibrated_clf.predict_proba(X_win_test)[:, 1]
preds_test = (probs_test >= best_thresh).astype(int)
auc_test   = roc_auc_score(y_win_test, probs_test)
acc_test   = (preds_test == y_win_test).mean()

print("=" * 60)
print("  Held-out Test Results (trials 8 & 9)")
print("=" * 60)
print(f"  Windows : {X_win_test.shape[0]:,}")
print(f"  AUC     : {auc_test:.4f}")
print(f"  Accuracy: {acc_test:.4f}  (threshold={best_thresh:.3f})")
print()
print(classification_report(y_win_test, preds_test, target_names=["Closed", "Open"]))
print("=" * 60)

# Per-trial breakdown
print("\nPer-trial breakdown:")
for t in TEST_TRIALS:
    mask_t = g_win_test == t
    auc_t  = roc_auc_score(y_win_test[mask_t], probs_test[mask_t])
    acc_t  = (preds_test[mask_t] == y_win_test[mask_t]).mean()
    print(f"  trial {t}: AUC={auc_t:.4f}  Acc={acc_t:.4f}  "
          f"n={mask_t.sum():,}")

## 8 · Plots

In [ ]:
%matplotlib inline
# ── A: LOTO AUC per trial ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=DARK)

ax = axes[0]
ax.set_facecolor(CARD)
colours = [TEAL] * len(TRAIN_TRIALS)
ax.bar(TRAIN_TRIALS, loto_aucs, color=colours, edgecolor=EDGE, alpha=0.9)
ax.axhline(np.mean(loto_aucs), color=GOLD, lw=2, ls="--",
           label=f"Mean={np.mean(loto_aucs):.4f}")
ax.set_xticks(TRAIN_TRIALS)
ax.set_xticklabels([f"T{t}" for t in TRAIN_TRIALS], color=WHITE)
ax.set_ylabel("AUC", color=WHITE); ax.set_xlabel("Held-out trial", color=WHITE)
ax.set_title("LOTO CV — AUC per held-out trial", color=WHITE, fontweight="bold")
ax.tick_params(colors=WHITE); ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)
for sp in ax.spines.values(): sp.set_color(EDGE)

# ── B: Optuna history ────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor(CARD)
trials_df = study.trials_dataframe(attrs=("number", "value", "state"))
comp = trials_df[trials_df["state"] == "COMPLETE"].sort_values("number")
vals = comp["value"].values; nums = comp["number"].values
best_sf = np.maximum.accumulate(vals)
ax.scatter(nums, vals, color=TEAL, alpha=0.45, s=25, zorder=3, label="Trial AUC")
ax.plot(nums, best_sf, color=GOLD, lw=2.5, zorder=4, label="Best so far")
ax.axhline(best_sf[-1], color=CORAL, lw=1, ls="--",
           label=f"Best={best_sf[-1]:.4f}", alpha=0.8)
ax.set_xlabel("Trial", color=WHITE); ax.set_ylabel("Validation AUC", color=WHITE)
ax.set_title("Optuna Search History", color=WHITE, fontweight="bold")
ax.tick_params(colors=WHITE); ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)
for sp in ax.spines.values(): sp.set_color(EDGE)

plt.suptitle("Multi-Trial Causal RF", color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()

In [ ]:
# ── C: Test trial probability + label timeline ────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(18, 7), sharex=True, facecolor=DARK)
t_ax = np.arange(len(probs_test))

ax = axes[0]
ax.set_facecolor(CARD)
ax.plot(t_ax, probs_test, color=TEAL, lw=0.6, label="Calibrated P(open)")
ax.axhline(best_thresh, color=CORAL, lw=1.2, ls="--",
           label=f"Threshold={best_thresh:.3f}")
# shade trial boundaries
split = int((g_win_test == TEST_TRIALS[0]).sum())
ax.axvline(split, color=GOLD, lw=1.5, ls=":", label=f"Trial {TEST_TRIALS[1]} start")
ax.set_ylim(0, 1); ax.set_ylabel("P(open)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

ax = axes[1]
ax.set_facecolor(CARD)
ax.plot(t_ax, preds_test, drawstyle="steps-post", color=CORAL, lw=1.5, label="Predicted")
ax.plot(t_ax, y_win_test, drawstyle="steps-post", color=WHITE, lw=1.0,
        alpha=0.5, label="True")
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(-0.1, 1.1); ax.set_yticks([0, 1])
ax.set_ylabel("Label", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

ax = axes[2]
ax.set_facecolor(CARD)
errors = (preds_test != y_win_test).astype(float)
ax.fill_between(t_ax, errors, color=GOLD, alpha=0.7, step="post", label="Error")
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(0, 1.2); ax.set_yticks([])
ax.set_ylabel("Error", color=WHITE, fontsize=9)
ax.set_xlabel("Sample (trials 8 & 9 concatenated)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

fig.suptitle(
    f"Held-Out Test (trials 8 & 9) · AUC={auc_test:.4f}  Acc={acc_test:.4f}",
    color=WHITE, fontsize=13, fontweight="bold"
)
plt.tight_layout()

In [ ]:
# ── D: Feature importance heatmap (channel × lag) ─────────────────────────
imp_matrix   = importances.reshape(N_CHANNELS, T)
ch_importance = imp_matrix.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=DARK,
                         gridspec_kw={"width_ratios": [3, 1]})

ax = axes[0]
ax.set_facecolor(CARD)
for sp in ax.spines.values(): sp.set_color(EDGE)
im = ax.imshow(imp_matrix, aspect="auto", cmap="viridis", interpolation="nearest")
plt.colorbar(im, ax=ax, label="Gini Importance")
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(EEG_COLS, color=WHITE, fontsize=9)
ax.set_xlabel("Lag (samples before target,  0 = most recent)",
              color=WHITE, fontsize=10)
ax.set_title("Feature Importance — Channel × Lag",
             color=WHITE, fontsize=12, fontweight="bold")
ax.tick_params(colors=WHITE)
best_ch, best_lag = np.unravel_index(imp_matrix.argmax(), imp_matrix.shape)
ax.add_patch(plt.Rectangle((best_lag-.5, best_ch-.5), 1, 1,
                            fill=False, edgecolor=CORAL, lw=2.5))
ax.text(best_lag, best_ch-.6,
        f"peak: {EEG_COLS[best_ch]} lag={best_lag}",
        color=CORAL, fontsize=8, ha="center")

ax = axes[1]
ax.set_facecolor(CARD)
for sp in ax.spines.values(): sp.set_color(EDGE)
order = np.argsort(ch_importance)[::-1]
colours = [GOLD if i == order[0] else TEAL for i in range(N_CHANNELS)]
ax.barh(range(N_CHANNELS),
        ch_importance[order[::-1]],
        color=colours[::-1], edgecolor=EDGE, alpha=0.9)
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels([EEG_COLS[i] for i in order[::-1]],
                   color=WHITE, fontsize=9)
ax.set_xlabel("Total importance (sum over lags)", color=WHITE, fontsize=9)
ax.set_title("Per-Channel", color=WHITE, fontsize=12, fontweight="bold")
ax.tick_params(colors=WHITE); ax.xaxis.grid(True, color=EDGE)

plt.suptitle(f"Feature Importance (T={T}, trained on trials {TRAIN_TRIALS})",
             color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()

In [ ]:
# ── E: Confusion matrices — LOTO vs Test ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor=DARK)

for ax, y_true, y_pred, title in [
    (axes[0], y_win[mask],  preds_opt,  f"LOTO CV (trials {TRAIN_TRIALS})"),
    (axes[1], y_win_test,   preds_test, f"Held-out test (trials {TEST_TRIALS})"),
]:
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_color(EDGE)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Closed", "Open"]).plot(
        ax=ax, colorbar=False, cmap="YlOrRd"
    )
    ax.set_title(title, color=WHITE, fontsize=10, fontweight="bold")
    ax.tick_params(colors=WHITE)
    ax.xaxis.label.set_color(WHITE); ax.yaxis.label.set_color(WHITE)
    for txt in ax.texts: txt.set_color("black"); txt.set_fontsize(12)

plt.suptitle("Confusion Matrices", color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()

## 9 · Results Summary & Save Model

In [ ]:
top3 = np.argsort(ch_importance)[::-1][:3]
print("=" * 60)
print("  EEG Causal RF — Multi-Trial Results")
print("=" * 60)
print(f"  Train trials  : {TRAIN_TRIALS}")
print(f"  Test  trials  : {TEST_TRIALS}")
print(f"  Look-back T   : {T} samples  ({T/FS*1000:.0f} ms)")
print(f"  Features      : {N_CHANNELS} ch × {T} lags = {N_CHANNELS*T}")
print()
print(f"  Optuna best AUC (inner CV) : {study.best_value:.4f}")
print(f"  LOTO  AUC (outer CV)       : {np.mean(loto_aucs):.4f} ± {np.std(loto_aucs):.4f}")
print(f"  Optimal threshold          : {best_thresh:.3f}")
print(f"  LOTO  Accuracy @ thresh    : {acc_opt:.4f}")
print()
print(f"  TEST  AUC                  : {auc_test:.4f}")
print(f"  TEST  Accuracy @ thresh    : {acc_test:.4f}")
print()
print("  Top-3 channels (by total importance):")
for i in top3:
    print(f"    {EEG_COLS[i]:6s}  {ch_importance[i]:.4f}")
print("=" * 60)

In [ ]:
import joblib

SAVE_PATH = f"../data/subject{SUBJECT}/model"
os.makedirs(SAVE_PATH, exist_ok=True)

joblib.dump(best_clf,       f"{SAVE_PATH}/eeg_rf_model.joblib")
joblib.dump(calibrated_clf, f"{SAVE_PATH}/eeg_rf_calibrated.joblib")

params_to_save = {
    "T":                  int(T),
    "n_estimators":       int(n_estimators),
    "max_depth":          max_depth,
    "min_samples_split":  int(min_samples_split),
    "min_samples_leaf":   int(min_samples_leaf),
    "max_features":       max_features,
    "class_weight":       class_weight,
    "loto_auc_mean":      float(np.mean(loto_aucs)),
    "loto_auc_std":       float(np.std(loto_aucs)),
    "test_auc":           float(auc_test),
    "best_thresh":        best_thresh,
    "n_channels":         int(N_CHANNELS),
    "channel_names":      EEG_COLS,
    "fs":                 int(FS),
    "train_trials":       TRAIN_TRIALS,
    "test_trials":        TEST_TRIALS,
    "per_window_norm":    True,
}
with open(f"{SAVE_PATH}/eeg_rf_params.json", "w") as f:
    json.dump(params_to_save, f, indent=2)

print("Saved:")
print(f"  {SAVE_PATH}/eeg_rf_model.joblib")
print(f"  {SAVE_PATH}/eeg_rf_calibrated.joblib")
print(f"  {SAVE_PATH}/eeg_rf_params.json")